# 🏷️ Thử nghiệm Metadata Extraction từ Chunks Pháp Luật
**Mục tiêu:** Dùng Gemini để tự động gán nhãn metadata phong phú hơn cho từng chunk (loại xe, loại vi phạm, mức phạt tiền, mức phạt điểm, v.v.) ngoài metadata cấu trúc sẵn có của `HierarchicalLegalSplitter`.

In [ ]:
import os, sys, json
from dotenv import load_dotenv

PROJECT_ROOT = os.path.abspath(os.path.join(os.getcwd(), '..', '..'))
if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)

load_dotenv(os.path.join(PROJECT_ROOT, '.env'))

import google.generativeai as genai
from source.core.config import Settings

settings = Settings()
api_key = settings.api_key or os.getenv('API_KEY')
genai.configure(api_key=api_key)
model = genai.GenerativeModel('gemini-2.0-flash')
print("✅ Gemini đã kết nối!")

## 1. Load chunks từ file đã tạo

In [ ]:
CHUNKS_PATH = os.path.join(PROJECT_ROOT, 'Data', 'chunks', 'traffic_chunks.json')
with open(CHUNKS_PATH, 'r', encoding='utf-8') as f:
    all_chunks = json.load(f)

print(f"✅ Đã tải {len(all_chunks):,} chunks từ {CHUNKS_PATH}")
print(f"\n📌 Ví dụ metadata hiện tại của chunk đầu tiên:")
print(json.dumps(all_chunks[0]['metadata'], ensure_ascii=False, indent=2))

## 2. Định nghĩa Prompt Extraction Metadata

In [ ]:
EXTRACTION_PROMPT = """
Bạn là chuyên gia phân tích pháp luật giao thông. Phân tích đoạn văn bản pháp lý sau và trích xuất thông tin dưới dạng JSON.

Đoạn văn bản:
\"\"\"{content}\"\"\"

Trả về JSON với các trường sau (nếu không có thông tin thì để null):
{{
  "loai_xe": ["ô tô", "xe máy", "xe đạp"],  // danh sách loại phương tiện liên quan
  "loai_vi_pham": "mô tả ngắn về hành vi vi phạm",
  "muc_phat_tien_min": 0,  // số tiền phạt tối thiểu (VNĐ)
  "muc_phat_tien_max": 0,  // số tiền phạt tối đa (VNĐ)
  "phat_bo_sung": "mô tả hình phạt bổ sung nếu có",
  "chu_the_vi_pham": "người điều khiển/chủ xe/...",
  "chu_de": "nồng độ cồn/tốc độ/dừng đỗ/đèn tín hiệu/..."
}}

Chỉ trả về JSON, không giải thích gì thêm.
"""

def extract_metadata(chunk_content: str) -> dict:
    """Gọi Gemini để trích xuất metadata từ 1 chunk."""
    try:
        prompt = EXTRACTION_PROMPT.format(content=chunk_content)
        response = model.generate_content(prompt)
        text = response.text.strip().replace('```json', '').replace('```', '').strip()
        return json.loads(text)
    except Exception as e:
        return {"error": str(e)}

print("✅ Hàm extract_metadata đã sẵn sàng!")

## 3. Chạy thử nghiệm trên 5 chunks mẫu

In [ ]:
# Chọn các chunk từ NĐ 100 (liên quan xử phạt → nhiều thông tin để trích xuất nhất)
nd100_chunks = [
    c for c in all_chunks 
    if '100' in c['metadata'].get('ten_van_ban', '')
    and c['metadata'].get('type') == 'khoan'
][:5]

print(f"🧪 Đang chạy thử nghiệm trên {len(nd100_chunks)} chunks từ NĐ 100...\n")

enriched_chunks = []
for i, chunk in enumerate(nd100_chunks):
    print(f"[{i+1}/{len(nd100_chunks)}] Đang trích xuất... | Điều: {chunk['metadata']['dieu']} | Khoản: {chunk['metadata']['khoan']}")
    extracted = extract_metadata(chunk['content'])
    
    # Merge metadata gốc với metadata mới
    enriched = dict(chunk)
    enriched['metadata_enriched'] = extracted
    enriched_chunks.append(enriched)
    
    print(f"  Chủ đề   : {extracted.get('chu_de', 'N/A')}")
    print(f"  Vi phạm  : {extracted.get('loai_vi_pham', 'N/A')}")
    print(f"  Phạt tiền: {extracted.get('muc_phat_tien_min', 0):,} - {extracted.get('muc_phat_tien_max', 0):,} VNĐ")
    print(f"  Loại xe  : {extracted.get('loai_xe', [])}")
    print()

## 4. So sánh Metadata gốc vs Metadata đã làm giàu

In [ ]:
if enriched_chunks:
    sample = enriched_chunks[0]
    print("📋 METADATA GỐC (HierarchicalLegalSplitter):")
    print(json.dumps(sample['metadata'], ensure_ascii=False, indent=2))
    print("\n🏷️  METADATA ĐÃ LÀM GIÀU (Gemini):")
    print(json.dumps(sample['metadata_enriched'], ensure_ascii=False, indent=2))
    print("\n📝 Nội dung chunk:")
    print(sample['content'])

## 5. Lưu kết quả enriched ra file

In [ ]:
output_path = os.path.join(os.getcwd(), 'traffic_chunks_enriched_sample.json')
with open(output_path, 'w', encoding='utf-8') as f:
    json.dump(enriched_chunks, f, ensure_ascii=False, indent=2)

print(f"✅ Đã lưu {len(enriched_chunks)} enriched chunks sang: {output_path}")
print("\n💡 Gợi ý tiếp theo: Chạy toàn bộ corpus và nạp vào Qdrant với payload metadata đầy đủ hơn!")